# Upload a local JSONL corpus to Hugging Face

Small helper for future sessions: validate a JSONL file, attach a dataset
card, push it, then verify that it loads back.

Requires HF_TOKEN in the environment (never hardcode it).


In [ ]:
from huggingface_hub import HfApi
from datasets import load_dataset
import json, os
from pathlib import Path

api = HfApi()
REPO_PREFIX = "michaelowusuntim6"
LOCAL_JSONL = "datasets/formatted/code_review/train.jsonl"
REPO_NAME = "my-new-corpus-qwen35"


In [ ]:
path = Path(LOCAL_JSONL)
rows = 0
bad = 0
with path.open(encoding="utf-8") as fh:
    for i, line in enumerate(fh, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            bad += 1
            print("bad JSON on line", i); continue
        if not isinstance(rec.get("messages"), list) or len(rec["messages"]) < 2:
            bad += 1
        rows += 1
print(f"{path}: {rows} records, {bad} bad")
assert bad == 0, "fix the data before uploading"


In [ ]:
card = f'''---
language:
  - en
license: other
task_categories:
  - text-generation
---
# {REPO_NAME}

Canonical Qwen3.5 chat records ({{"messages": [...]}}), consumed via
tokenizer.apply_chat_template().
'''
(Path(LOCAL_JSONL).parent / "README.md").write_text(card, encoding="utf-8")


In [ ]:
token = os.environ["HF_TOKEN"]
repo_id = f"{REPO_PREFIX}/{REPO_NAME}"
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=str(Path(LOCAL_JSONL).parent),
                  repo_type="dataset", commit_message=f"upload {REPO_NAME}")
print("https://huggingface.co/datasets/" + repo_id)


In [ ]:
ds = load_dataset(repo_id, split="train")
print(len(ds), "records;", "messages" in ds[0])
print(ds[0]["messages"][1]["content"][:200])
